In [86]:
import yfinance as yf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import numpy as np

In [87]:
end_date = '2024-10-15'
stock_data = yf.download('^GSPC', start='2019-10-15', end=end_date)

[*********************100%***********************]  1 of 1 completed


In [88]:
#Download additional data for 10 days after the initial end date

additional_data = yf.download('^GSPC', start=pd.to_datetime(end_date) + pd.Timedelta(days=1), 
end=pd.to_datetime(end_date) + pd.Timedelta(days=10))

[*********************100%***********************]  1 of 1 completed


In [89]:
# Step 2: Feature Engineering
# Calculate daily returns
stock_data['Return'] = stock_data['Adj Close'].pct_change()
additional_data['Return'] = additional_data['Adj Close'].pct_change()

In [90]:
# Calculate rolling volatility (standard deviation of returns over a window)
stock_data['Volatility'] = stock_data['Return'].rolling(window=21).std()  # 21-day rolling volatility
additional_data['Volatility'] = additional_data['Return'].rolling(window=21).std()

In [91]:
# Drop NaN values
stock_data.dropna(inplace=True)
additional_data.dropna(inplace=True)

# Step 3: Feature Selection
# We use lagged returns and volatility as features
stock_data['Lagged_Volatility'] = stock_data['Volatility'].shift(1)
stock_data['Lagged_Return'] = stock_data['Return'].shift(1)
additional_data['Lagged_Volatility'] = additional_data['Volatility'].shift(1)
additional_data['Lagged_Return'] = additional_data['Return'].shift(1)


In [92]:
# Prepare the features and target for modeling
X = stock_data[['Lagged_Volatility', 'Lagged_Return']]
y = stock_data['Volatility']

# Drop NaN values that result from shifting
X = X.dropna()
y = y.loc[X.index]

In [93]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [94]:
# Step 4: Model Training using KNN
knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_train, y_train)

# Step 5: Volatility Prediction and Evaluation
y_pred = knn_model.predict(X_test)
mse = root_mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error for Volatility Prediction: {mse}')


Mean Squared Error for Volatility Prediction: 0.0005405383212237252


In [95]:
import warnings
warnings.filterwarnings('ignore')

# Step 6: Forecasting Future Volatility
forecast_horizon = 10  # Number of days to forecast
forecasted_volatility = []
forecast_dates = pd.date_range(start=X_test.index[-1] + pd.Timedelta(days=1), periods=forecast_horizon, freq='B')

last_X = X_test.iloc[-1, :].values.reshape(1, -1)  # Take the last row from X_test

for _ in range(forecast_horizon):
    # Predict the next volatility
    next_volatility = knn_model.predict(last_X)
    forecasted_volatility.append(next_volatility[0])

    # Update last_X with the predicted volatility and simulate a small random return
    new_lagged_volatility = next_volatility[0]
    new_lagged_return = np.random.normal(0, np.std(X_test['Lagged_Return']))  # Simulate return with noise
    last_X = np.array([[new_lagged_volatility, new_lagged_return]])

# Convert forecasted data to DataFrame for easy plotting
forecast_df = pd.DataFrame({
    'Date': forecast_dates,
    'Forecasted_Volatility': forecasted_volatility
})

In [98]:
# True data plot steps
end_date_true_plot = '2024-10-18'
stock_data_true = yf.download('^GSPC', start='2023-01-01', end=end_date_true_plot)
stock_data_true['Return'] = stock_data_true['Adj Close'].pct_change()
stock_data_true['Volatility'] = stock_data_true['Return'].rolling(window=21).std()
stock_data_true.dropna(inplace=True)


[*********************100%***********************]  1 of 1 completed


In [100]:
from plotly.subplots import make_subplots
import plotly.graph_objs as go

# Create a figure
fig = make_subplots()

# Add predicted volatility trace
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_pred * 100,  # Convert to percentage
    mode='lines', 
    name='KNN Predicted Volatility', 
    line=dict(color='green', dash='solid'),
    opacity=0.5
))

# Add forecasted volatility trace
fig.add_trace(go.Scatter(
    x=forecast_df['Date'], 
    y=forecast_df['Forecasted_Volatility'] * 100,  # Convert to percentage
    mode='lines', 
    name='Forecasted Volatility', 
    line=dict(color='red', dash='dash'),
    opacity=0.7
))

# Add actual volatility trace
fig.add_trace(go.Scatter(
    x=stock_data_true.index, 
    y=stock_data_true['Volatility'] * 100,  # Convert to percentage
    mode='lines', 
    name='Actual Volatility', 
    line=dict(color='blue', dash='solid')
))

# Update layout
fig.update_layout(
    title='SP500 Stock Volatility: Actual, Predicted, and Forecast',
    xaxis_title='Date',
    yaxis_title='Volatility (%)',  # Update Y axis title
    legend=dict(x=0, y=1)
)

# Show the figure
fig.show()
